# World Mechanics — Containers & Action Durations (issues #43, #24)

Two **opt-in** world-modeling features. Both are off by default, so simple games stay
simple:

- **Containers & carry capacity (#43):** characters have a limited number of *hand slots*,
  and items can be *contained* (a backpack holds more without filling your hands).
- **Action durations & a per-turn budget (#24):** actions can declare how many in-game
  minutes they take, so with a clock an NPC can squeeze several cheap actions into one turn.

Fully **offline and deterministic**. Builds on the engine basics (`01_engine_tutorial.ipynb`)
and the game clock (`02_agents_react.ipynb`).

## 1. Containers

Any item becomes a container with `make_container(capacity=...)`. It then tracks its
`contents`, reports `current_count()`, and knows when it `is_full()`. A `capacity` of
`None` means unlimited.

In [1]:
from text_adventure_games import things

pack = things.Item("backpack", "a sturdy backpack")
pack.make_container(capacity=2)
print("is_container:", pack.get_property("is_container"), "| capacity:", pack.capacity)

pack.add_item(things.Item("rock", "a plain rock"))
print("count:", pack.current_count(), "| has_space:", pack.has_space())

pack.add_item(things.Item("gem", "a shiny gem"))
print("count:", pack.current_count(), "| is_full:", pack.is_full())

is_container: True | capacity: 2
count: 1 | has_space: True
count: 2 | is_full: True


## 2. Carry capacity (hand slots)

`Character.carry_capacity` limits how many items the character can hold in hand. The
default is `None` (unlimited — backward compatible). When hands are full, a carried
container absorbs the overflow: `accept_item()` tries hands first, then any container.

In [2]:
player = things.Character("player", "the player", "I explore.")
print("default carry_capacity:", player.carry_capacity, "| has_hand_space:", player.has_hand_space())

player.carry_capacity = 1
pack = things.Item("backpack", "a sturdy backpack")
pack.make_container(capacity=2)
player.add_to_inventory(pack)  # the single hand slot is now full
print("hands full after holding the pack?", not player.has_hand_space())

placed = player.accept_item(things.Item("rock", "a plain rock"))
print("rock accepted via overflow:", placed, "| stowed in pack:", "rock" in pack.contents)

default carry_capacity: None | has_hand_space: True
hands full after holding the pack? True
rock accepted via overflow: True | stowed in pack: True


## 3. `Get` respects capacity

The `Get` action runs all of this behind the precondition gate. With one full hand and a
pack with room, a picked-up item is **stowed in the pack**. When hands *and* containers are
full, `Get` **fails gracefully** — the world is left unchanged and the failure reason is
recorded.

In [3]:
from text_adventure_games import games
from text_adventure_games.actions import things as thing_actions
from text_adventure_games.reporting import CaptureRenderer, Channel

room = things.Location("Room", "A plain room.")
player = things.Character("player", "the player", "I explore.")
player.carry_capacity = 1
game = games.Game(room, player, characters=[])
game.parser.set_renderer(CaptureRenderer())

pack = things.Item("backpack", "a sturdy backpack")
pack.make_container(capacity=1)
player.add_to_inventory(pack)  # fills the one hand slot
room.add_item(things.Item("rock", "a plain rock"))

thing_actions.Get(game, "get rock", actor=player)()
print("rock stowed in the pack:", "rock" in pack.contents)

# Now hands AND the pack are full; a second pickup fails at the gate.
room.add_item(things.Item("gem", "a shiny gem"))
action = thing_actions.Get(game, "get gem", actor=player)
print("can pick up the gem?", action.check_preconditions())
print("reason:", game.parser.last_fail_message)
print("gem still in the room:", "gem" in room.items)

rock stowed in the pack: True
can pick up the gem? False
reason: Your hands are full and you have nothing with room to stow it.
gem still in the room: True


## 4. Inventory shows contents & counts

The `inventory` action lists carried containers with a `(count/capacity)` tag and shows
their nested contents.

In [4]:
room = things.Location("Room", "A plain room.")
player = things.Character("player", "the player", "I explore.")
game = games.Game(room, player, characters=[])
cap = CaptureRenderer()
game.parser.set_renderer(cap)

pack = things.Item("backpack", "a sturdy backpack")
pack.make_container(capacity=5)
player.add_to_inventory(pack)
pack.add_item(things.Item("rock", "a plain rock"))
pack.add_item(things.Item("gem", "a shiny gem"))

thing_actions.Inventory(game, "inventory", actor=player)()
print("\n".join(cap.texts(Channel.NARRATION)))

player's inventory contains:
* a sturdy backpack (2/5)
    - a plain rock
    - a shiny gem



## 5. Action durations & the per-turn budget

An action can declare `DURATION` — the in-game minutes it costs. Quick sensory/informational
actions are cheap; movement stays at the default so the clock stays predictable.

In [5]:
from text_adventure_games.actions.things import Examine, Inventory
from text_adventure_games.actions.rose import Smell_Rose
from text_adventure_games.actions.locations import Go

print("Smell_Rose.DURATION:", Smell_Rose.DURATION)
print("Examine.DURATION:   ", Examine.DURATION)
print("Inventory.DURATION: ", Inventory.DURATION)
print("Go.DURATION:        ", Go.DURATION, "(default -- not pinned to a cost)")

Smell_Rose.DURATION: 1
Examine.DURATION:    1
Inventory.DURATION:  1
Go.DURATION:         None (default -- not pinned to a cost)


## 6. Several cheap actions in one turn

Give the game a `GameClock` with `minutes_per_turn`, and an NPC spends that budget across
as many actions as fit. Here, three 5-minute moves all happen inside one 15-minute turn.
(Driven by a scripted `MockLlmClient` that returns a `Duration:` line with each action.)

In [6]:
from text_adventure_games.clock import GameClock
from text_adventure_games.llm_client import MockLlmClient
from text_adventure_games.npc import make_react_behavior

field = things.Location("Field", "An open grassy field.")
forest = things.Location("Forest", "A dark tangled forest.")
field.add_connection("north", forest)
player = things.Character("player", "a brave adventurer", "I explore.")
troll = things.Character("troll", "a mean green troll", "I am hungry.")
game = games.Game(field, player, characters=[troll])
field.add_character(troll)
game.clock = GameClock(minutes_per_turn=15)

mock = MockLlmClient(
    [
        "Action: go north\nDuration: 5",
        "Action: go south\nDuration: 5",
        "Action: go north\nDuration: 5",
    ]
)
troll.set_behavior(make_react_behavior(mock))

troll.take_turn(game)
print("actions taken this turn:", len(mock.calls), "(three 5-min moves fit a 15-min turn)")
print("troll ended in:", troll.location.name)

troll [action] go north
troll moved to Forest
FIELD
(8:00 AM (morning))
An open grassy field.
Exits:
 * North to Forest




troll [action] go south
troll moved to Field
FIELD
(8:00 AM (morning))
An open grassy field.
Exits:
 * North to Forest


Characters:
 * troll - a mean green troll

troll [action] go north
troll moved to Forest
FIELD
(8:00 AM (morning))
An open grassy field.
Exits:
 * North to Forest




actions taken this turn: 3 (three 5-min moves fit a 15-min turn)
troll ended in: Forest


### Takeaways

- Both features are **opt-in**: no `carry_capacity` → unlimited hands; no clock → exactly
  one action per turn (the pre-#24 behavior).
- Containers route overflow automatically; `Get`/`Drop`/`Give` all respect capacity through
  the precondition gate.
- Durations let an NPC bundle cheap actions; a safety cap (`MAX_ACTIONS_PER_TURN`) prevents
  runaway loops. The clock itself is introduced in `02_agents_react.ipynb`.
- See `tests/test_containers.py` and `tests/test_action_durations.py` for the full specs.